In [2]:
import pandas as pd
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

In [3]:
# import pandas as pd
# df = pd.read_csv('/content/monks-3-new.csv')

In [4]:
# # y=df['Class']
# # df_with_class=df.copy()
# # df.drop(['Class'],axis=1,inplace=True)
# # X=df
# import pandas as pd
# import random

# # -------- Load data --------
# df = pd.read_csv('./monks-1-new.csv')

# # -------- Separate class column --------
# class_col = 'Class'
# attributes = [col for col in df.columns if col != class_col]

# # -------- Shuffle attributes --------
# random.shuffle(attributes)

# # -------- Rebuild dataframe --------
# new_order = [class_col] + attributes
# df_shuffled = df[new_order]

# # -------- Split X and y --------
# y = df_shuffled['Class']
# X = df_shuffled.drop(['Class'], axis=1)

# print("New column order:")
# print(df_shuffled.columns)

In [5]:
import pandas as pd

df = pd.read_csv('./monks-1-new.csv')

# ✅ Fixed column order
new_order = ['Class', 'A5', 'A4', 'A6', 'A1', 'A2', 'A3']

df_fixed = df[new_order]

y = df_fixed['Class']
X = df_fixed.drop(['Class'], axis=1)

print(df_fixed.columns)

Index(['Class', 'A5', 'A4', 'A6', 'A1', 'A2', 'A3'], dtype='str')


In [6]:
dict_x=X.to_dict()

In [7]:
from math import ceil
from math import floor

import pandas as pd



def check_type(variable):
    if isinstance(variable, (int, float)):
        return "Number"
    elif isinstance(variable, str):
        return "String"
    else:
        return "Other type"




def frequency_encoding(data):
    bits = 3
    encoder = ['000', '001', '011', '111']  # for numeric encoding

    # Define categorical encodings
    categorical_map = {
        "yesno": {"yes": "01", "no": "10"},
        "lowhigh": {"low": "01", "high": "10"},
        "lowmedhigh": {"low": "001", "medium": "010", "high": "100"}
    }

    for col in data:
        first_val = str(data[col][0]).strip().lower()

        # --- Handle Yes/No ---
        if first_val in ["yes", "no"]:
            for i, val in data[col].items():
                val_str = str(val).strip().lower()
                data[col][i] = categorical_map["yesno"].get(val_str, "00")

        # --- Handle Low/Medium/High ---
        elif first_val in ["low", "high"]:
            for i, val in data[col].items():
                val_str = str(val).strip().lower()
                data[col][i] = categorical_map["lowhigh"].get(val_str, "00")

        elif first_val in ["low", "medium", "high"]:
            for i, val in data[col].items():
                val_str = str(val).strip().lower()
                data[col][i] = categorical_map["lowmedhigh"].get(val_str, "000")

        # --- Handle numeric/string frequency encoding ---
        elif check_type(data[col][0]) in ["String", "Number"]:
            col_arr = list(data[col].values())
            col_arr.sort()
            t = int(len(col_arr) / (bits + 1))
            r = len(col_arr) % (bits + 1)
            index_map = {}
            j = 0
            for i in range(len(col_arr)):
                if (i + 1) % t == 0:
                    index_map[col_arr[i]] = j
                    j += 1
                    j = min(j, 3)
                else:
                    index_map[col_arr[i]] = j
            for i, (key, value) in enumerate(data[col].items()):
                data[col][i] = encoder[index_map[value]]

    return data

In [8]:
data=frequency_encoding(dict_x)

In [9]:
X=pd.DataFrame(data)
y=LabelEncoder().fit_transform(y)

In [10]:
X['Target Configuration']=X.apply(lambda row: ''.join(map(str, row)), axis=1)

In [11]:
X

,A5,A4,A6,A1,A2,A3,Target Configuration
0,000,001,001,001,001,001,000001001001001001
1,000,001,111,001,001,001,000001111001001001
2,001,001,001,001,001,001,001001001001001001
3,001,001,111,001,001,001,001001111001001001
4,011,001,001,001,001,001,011001001001001001
...,...,...,...,...,...,...,...
427,001,111,111,111,111,111,001111111111111111
428,011,111,001,111,111,111,011111001111111111
429,011,111,111,111,111,111,011111111111111111
430,111,111,001,111,111,111,111111001111111111


In [12]:
# import pandas as pd
# Rename columns in X
X = X.rename(columns={
    "Feature_1": "Encoded_Feature_1",
    "Feature_2": "Encoded_Feature_2"
})
# Reset indexes before concatenation to avoid NaN
df_final = pd.concat([df.reset_index(drop=True),
                      X.reset_index(drop=True)],
                      axis=1)

# Add target as last column
df_final["target"] = y

print(df_final.to_string(line_width=200))
# print(df_final)

     Class  A5  A1  A2  A4  A3  A6   A5   A4   A6   A1   A2   A3 Target Configuration  target
0        1   1   1   1   1   1   1  000  001  001  001  001  001   000001001001001001       1
1        1   1   1   1   1   1   2  000  001  111  001  001  001   000001111001001001       1
2        1   2   1   1   1   1   1  001  001  001  001  001  001   001001001001001001       1
3        1   2   1   1   1   1   2  001  001  111  001  001  001   001001111001001001       1
4        1   3   1   1   1   1   1  011  001  001  001  001  001   011001001001001001       1
5        1   3   1   1   1   1   2  011  001  111  001  001  001   011001111001001001       1
6        1   4   1   1   1   1   1  111  001  001  001  001  001   111001001001001001       1
7        1   4   1   1   1   1   2  111  001  111  001  001  001   111001111001001001       1
8        1   1   1   1   2   1   1  000  011  001  001  001  001   000011001001001001       1
9        1   1   1   1   2   1   2  000  011  111  001  001 

In [13]:
import math
def lcm(a, b):
    return abs(a * b) // math.gcd(a, b)

def pad_configurations(training_data, n):
    padded_data = {}
    #target_multiple = lcm(n, 3)  # Make length a multiple of both n and 3
    target_multiple = n
    for config, label in training_data.items():
        length = len(config)
        remainder = length % target_multiple
        if remainder != 0:
            padding = target_multiple - remainder
            config = '0' * padding + config  # prepend zeros
        padded_data[config] = label
    # Print length of one padded config
    example_config = next(iter(padded_data))
    #print("length of padded data =", len(example_config))
    return padded_data

In [14]:
import math

def find_optimal_n(training_data):
    if not training_data:
        return None  # no data

    l = len(next(iter(training_data)))   # length of any one config
    num_instances = len(training_data)
    #print("length of configuration",l)
    for s in range(2, 22):  # candidate split sizes 1..21
        chunks = math.ceil(l / s)             # chunks per config
        #print("Segments",chunks, "each with split size ",s)
        total_chunks = chunks * num_instances # total across all configs
        #print("Before split number of instances",num_instances)
        #print("After split number of instances",total_chunks)
        # Condition
        if total_chunks < 2 ** s:
            #print("Satisfied n is",s)
            return s
        else:
            #print("Split size not optimal",s)
            print("\n")



    return None  # No suitable split size found


In [15]:
import random
def solve(number, rulenumber):
    if ((1 << number) & rulenumber) > 0:
        return 1
    return 0




def rule_to_binary(rule, num_states):
    return format(rule, '0' + str(num_states**3) + 'b')

def evolve_cell(left, center, right, rule_number):
    index = int(str(left)+str(center)+str(right), 2)
    if((1<<index)&rule_number)>0:
        return '1'
    return '0'

def evolve_ca(row, rule_numbers):
    num_cells = len(row)
    next_row =""

    for i in range(num_cells):
        if(i-1<0):
            left=0
        else:
            left = int(row[(i - 1) % num_cells])
        center = int(row[i])
        if(i+1==num_cells):
            right=0
        else:
            right = int(row[(i + 1) % num_cells])
        next_row+= evolve_cell(left, center, right, rule_numbers[i])
    return next_row

def binary_to_decimal(binary):
    decimal=int(binary,2)
    return decimal

def decimal_to_binary(number,n):
    binary=bin(number)
    binary=binary[2:]
    if len(binary) is not n:
        for i in range(n-len(binary)):
            binary='0'+binary
    return binary

def binatodeci(binary):
    return int(binary,2)

def generate_ca(number, rule_numbers, all_numbers,n):
    initial_row=decimal_to_binary(number,n)
    ca_history = [initial_row]
    #print("Intial configuration",initial_row)
    s=set()
    all_numbers[number]=1
    while True:
        new_row = evolve_ca(ca_history[-1], rule_numbers)
        #print("Next configuration",new_row)
        if new_row in s:
            return -1
        if new_row==initial_row:
            ca_history.append(new_row)
            break
        s.add(new_row)
        ca_history.append(new_row)
        # print(new_row)
        number=binatodeci(new_row)
        all_numbers[number]=1
    return ca_history

def get_index(arr):
    return arr[1]

def classification(data,testing_data,n_op,rule_numbers):
    #print("Cell Length:",n)
    t=2**n_op



    all_numbers=[0]*t
    ca_history=[]
    for i in range(t):
        if(all_numbers[i]==0):
            history = generate_ca(i, rule_numbers,all_numbers,n_op)
            #print("cycles generating:",history)
            ca_history.append(history)
            if history==-1:
                ca_history=[]
                break

    map_to_cycle={}

    #print(ca_history)
    #Number_of_cycles=len(ca_history)
    # print(Number_of_cycles)
    # Length of ca_history
    xor_values_for_each_cycle = []   # Stores [cycle_index, xor_value] for each cycle

    for i in range(len(ca_history)):   # Iterate over each cycle in the CA history
        xor_label_index_dict = []      # Temporary list to hold [cycle_index, xor_value]
        xor_a = 0

        for j in range(len(ca_history[i])):   # Iterate over each configuration in cycle i
            num = binary_to_decimal(ca_history[i][j])   # Convert binary configuration to decimal
            xor_a = xor_a ^ num                         # Update XOR value with current
            map_to_cycle[ca_history[i][j]] = i          # Map configuration to its cycle index

        xor_label_index_dict.append(i)      # Add cycle index
        xor_label_index_dict.append(xor_a)  # Add final XOR value for this cycle

        xor_values_for_each_cycle.append(xor_label_index_dict)  # Store result

    # print(1)

    # First, find number of classes (maximum label + 1)
    num_classes = max(data.values()) + 1

    # Initialize count dictionary with zeroes for all classes
    count_majority_for_each_cycle = {}
    for i in range(len(ca_history)):
        count_majority_for_each_cycle[i] = [0] * num_classes
    #print("Count_Majority in cycle",count_majority_for_each_cycle)

    # Count how many configs of each label fall in each cycle
    #print("data",data)

    #print("Cycle:",map_to_cycle)
    for config, label in data.items():
        #print("config and cycle config",map_to_cycle)
        cycle_index = map_to_cycle[config]
        count_majority_for_each_cycle[cycle_index][label] += 1
    #print("Count_Majority in cycle",count_majority_for_each_cycle)
    #print("Rule:", rule_numbers)
    cycle_label={}
    # print("\nAssigned cycle labels:")
    # for cycle_id, label in cycle_label.items():
    #     print(f"Cycle {cycle_id}: Label = {label}")
    for i, (key, value) in enumerate(count_majority_for_each_cycle.items()):
        # i = index of the cycle
        # key = cycle id (e.g., 0, 1, 2, ...)
        # value = list of counts of class labels for this cycle

        label = value.index(max(value))
        # Find the label with the maximum count (majority label)

        c = 0
        for j in range(len(value)):
            if (value[label] == value[j]):
                c += 1
        # Count how many labels have the same max frequency (to detect ties)

        # Generate a unique conflict label ID
        conflict_label = max(
            set(label for label_list in count_majority_for_each_cycle.values()
                for label in range(len(label_list)))
        ) + 1
        # This ensures conflict labels are different from normal labels

        if c > 1:
            # If there is a tie (multiple labels with same max count)
            cycle_label[i] = conflict_label
            xor_values_for_each_cycle[i].append(conflict_label)
        else:
            # Otherwise, assign the majority label normally
            cycle_label[i] = label
            xor_values_for_each_cycle[i].append(label)


    # STEP 1: Sort based on some feature (XOR values )
    sorted_array = sorted(xor_values_for_each_cycle, key=get_index)
    #print("Rule:", rule_numbers)


    # STEP 2: Identify large gaps in index values to segment clusters
    gap_index_arr = [0]
    # Stores the indices where a "gap" is detected in sorted_array
    # Start with 0 (beginning of array)

    curr_diff = sorted_array[1][1] - sorted_array[0][1]
    # Initial difference between first two values (second column of sorted_array)

    for i in range(2, len(sorted_array)):
        diff = sorted_array[i][1] - sorted_array[i-1][1]
        # Compute difference between consecutive values

        if curr_diff == 0:
            curr_diff = diff
            # If current difference is 0, update it with new diff (to avoid divide-by-zero)

        elif diff > 10 * curr_diff:
            # If the new difference is much larger (10x threshold),
            # treat this as a "gap" → mark index i
            gap_index_arr.append(i)
            curr_diff = 0
            # Reset curr_diff so next comparison starts fresh
            continue

        curr_diff = diff
        # Update current difference for next iteration

    gap_index_arr.append(len(sorted_array))
# Always add the last index (end boundary)




    # STEP 3: Count labels in each cluster and resolve conflict (label == conflict_label)
    conflict_label = max([x[2] for x in sorted_array])  # assumed to be the last label (e.g., if k=4, conflict_label=4)
    num_classes = conflict_label  # excluding the conflict label

    j = 1
    label_count = [0] * num_classes  # dynamically sized
    for i in range(len(sorted_array) + 1):
        if i == gap_index_arr[j]:
            # determine the label with max frequency
            max_val = max(label_count)
            tied_labels = [i for i, count in enumerate(label_count) if count == max_val]

            # Check if tie
            if len(tied_labels) == 1:
                max_label = tied_labels[0]
            else:
                # Tie-break: use training data present in this cluster
                class_counts_in_cluster = {label: 0 for label in tied_labels}

                # Iterate through cycles in current cluster
                for k in range(gap_index_arr[j-1], gap_index_arr[j]):
                    cycle_id = sorted_array[k][0]
                    if cycle_id in count_majority_for_each_cycle:
                        for label in tied_labels:
                            class_counts_in_cluster[label] += count_majority_for_each_cycle[cycle_id][label]

                # Choose the label with highest count from training data
                max_label = max(class_counts_in_cluster, key=class_counts_in_cluster.get)

            # resolve conflicts in this cluster
            for k in range(gap_index_arr[j-1], gap_index_arr[j]):
                if sorted_array[k][2] == conflict_label:
                    cycle_id = sorted_array[k][0]
                    cycle_label[cycle_id] = max_label
                    sorted_array[k][2] = max_label
            # reset counters for next cluster
            label_count = [0] * num_classes
            j += 1

        if i < len(sorted_array):
            label = sorted_array[i][2]
            if label != conflict_label:
                label_count[label] += 1




    Cycle_representative_class_label = {}

    # pick first configuration from each cycle as representative
    for cycle_id, label in cycle_label.items():
        if cycle_id < len(ca_history) and ca_history[cycle_id]:
            representative_config = ca_history[cycle_id][0]  # first config in cycle
            Cycle_representative_class_label[representative_config] = label

    #print("Cycle_representative_class_label:", Cycle_representative_class_label)



    from sklearn.metrics import precision_score, recall_score, f1_score

# --- Testing ---
    count = 0
    y_true = []
    y_pred = []

    for i, (config, true_label) in enumerate(testing_data.items()):
        # Step 1: Split the configuration into n-sized chunks
        chunks = [config[j:j+n_op] for j in range(0, len(config), n_op)]
        chunk_labels = []

        for chunk in chunks:
            start_config = chunk
            current_config = chunk
            predicted = None

            while True:
                # evolve one step
                next_config = evolve_ca(current_config, rule_numbers)

                # check if next config has a representative label
                if next_config in Cycle_representative_class_label:
                    predicted = Cycle_representative_class_label[next_config]
                    break

                # stop if we came back to start (no representative found)
                if next_config == start_config:
                    break

                # move forward
                current_config = next_config

            if predicted is not None:
                chunk_labels.append(predicted)

        # Step 2: Majority vote from chunk labels
        if chunk_labels:
            predicted_label = max(set(chunk_labels), key=chunk_labels.count)

            # Step 3: Compare with true label
            y_true.append(true_label)
            y_pred.append(predicted_label)

            if predicted_label == true_label:
                count += 1

    # --- Metrics ---
    accuracy = count / len(y_true) if y_true else 0.0
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    # --- Results ---
    result = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "rule_numbers": rule_numbers.copy(),
        "Number_of_cycles": len(ca_history),
        "Cycle_lengths": [len(inner) - 1 for inner in ca_history]
    }

    return result



In [16]:
import random
import time
import pandas as pd
from itertools import product
from sklearn.model_selection import train_test_split

# =========================
# TEMPLATE EXPANSION (FIXED)
# =========================
def generate_rules_from_template(template, n):

    # ✅ extract integers (IMPORTANT FIX)
    fixed_prefix = [template[0][0], template[1][0]]

    repeat_set = template[2]     # {60,195}
    mid_set = template[3]        # {53,58,83,163}
    last_set = template[4]       # {20,65}

    rules = []

    repeat_len = n - 4   # because 2 fixed + 1 mid + 1 last

    if repeat_len < 0:
        return []

    # generate repeat combinations
    repeat_combinations = list(product(repeat_set, repeat=repeat_len)) if repeat_len > 0 else [()]

    for r_combo in repeat_combinations:
        for mid in mid_set:
            for last in last_set:

                rule = []
                rule.extend(fixed_prefix)
                rule.extend(r_combo)
                rule.append(mid)
                rule.append(last)

                rules.append(rule)

    return rules


# =========================
# ALL RULES FROM TEMPLATES
# =========================
def get_all_rules_from_templates(n):

    templates = [
        # [[5], [15], [60,195], [53,58,83,163], [20,65]],
        # [[10], [240], [60,195], [53,58,83,163], [20,65]],
        [[5], [15], [60, 195], [92, 172, 197, 202], [20, 65]],
        [[10], [240], [60, 195], [92, 172, 197, 202], [20, 65]]

    ]

    all_rules = []

    for t in templates:
        rules = generate_rules_from_template(t, n)
        all_rules.extend(rules)

    return all_rules


# =========================
# MAIN EXPERIMENT
# =========================
max_ans = 0

# -------- Prepare data --------
y = pd.Series(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

X_train = X_train.to_dict()

training_data = {}
for i, (key, value) in enumerate(X_train['Target Configuration'].items()):
    training_data[value] = y_train.iloc[i]

# -------- Find optimal n --------
n_opt = find_optimal_n(training_data)
print("Optimal n:", n_opt)

# -------- Padding --------
training_data = pad_configurations(training_data, n_opt)

new_training_data = {}
for config, label in training_data.items():
    for i in range(0, len(config), n_opt):
        chunk = config[i:i + n_opt]
        new_training_data[chunk] = label

# -------- Testing data --------
testing_data = {}
for i, (key, value) in enumerate(X_test['Target Configuration'].items()):
    testing_data[value] = y_test.iloc[i]

testing_data = pad_configurations(testing_data, n_opt)

# =========================
# GENERATE RULES
# =========================
# all_rule_vectors = get_all_rules_from_templates(n_opt)
all_rule_vectors=[[10, 240, 195, 195, 195, 60, 195, 195, 92, 65]]

print("Total generated rules:", len(all_rule_vectors))

# =========================
# TEST EACH RULE
# =========================
for rule_numbers in all_rule_vectors:

    # ✅ SAFETY CHECK (avoid your previous error)
    if not all(isinstance(x, int) for x in rule_numbers):
        continue

    start_time = time.time()

    result = classification(new_training_data, testing_data, n_opt, rule_numbers)

    execution_time = time.time() - start_time

    print("Rule:", rule_numbers)
    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1-Score : {result['f1_score']:.4f}")
    print(f"Time     : {execution_time:.2f}s")
    print("-"*40)

    if max_ans < result['accuracy']:
        max_ans = result['accuracy']

    # -------- SAVE TO FILE --------
    with open("monks-1-template-swap-results.txt", "a") as f:
        f.write(
            f"{rule_numbers} | "
            f"Acc: {result['accuracy']:.4f} | "
            f"P: {result['precision']:.4f} | "
            f"R: {result['recall']:.4f} | "
            f"F1: {result['f1_score']:.4f}\n"
        )

print("\nMaximum Accuracy =", max_ans)


















Optimal n: 10
Total generated rules: 1
Rule: [10, 240, 195, 195, 195, 60, 195, 195, 92, 65]
Accuracy : 0.8851
Precision: 0.8951
Recall   : 0.8787
F1-Score : 0.8824
Time     : 0.12s
----------------------------------------

Maximum Accuracy = 0.8850574712643678


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# --- Split dataset (keep as DataFrame/array, not dict!) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

# --- Models ---
models = {
    "SVM": SVC(kernel="rbf", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    # "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
}

# --- Training & Evaluation ---
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)  # ✅ Works now (X_train is numeric DataFrame/array)
    y_pred = model.predict(X_test)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }

# --- Print Results ---
for name, metrics in results.items():
    print(f"\n{name} Results:")
    print(f"  Accuracy : {metrics['Accuracy']:.4f}")
    print(f"  Precision: {metrics['Precision']:.4f}")
    print(f"  Recall   : {metrics['Recall']:.4f}")
    print(f"  F1-score : {metrics['F1-score']:.4f}")



SVM Results:
  Accuracy : 0.5632
  Precision: 0.5591
  Recall   : 0.5585
  F1-score : 0.5585

KNN Results:
  Accuracy : 0.7126
  Precision: 0.8264
  Recall   : 0.6875
  F1-score : 0.6677

Random Forest Results:
  Accuracy : 0.9195
  Precision: 0.9280
  Recall   : 0.9144
  F1-score : 0.9180

Decision Tree Results:
  Accuracy : 0.9425
  Precision: 0.9519
  Recall   : 0.9375
  F1-score : 0.9414
